# Vocabulary Evaluator (Early Release)

**The Vocabulary Evaluator** gives developers the fine-grained insight they need but can’t get from traditional tools. It helps determine whether texts use words that align with grade-level expectations and support growth in academic language. This ensures students are consistently exposed to the kinds of vocabulary that build knowledge and enable them to fully engage with grade-level texts.

By understanding what makes a text difficult for a student to read, edtech companies and educators are better equipped to ensure students get the right text for their needs, along with the right instructional supports.

You can use this evaluator to help ensure AI-generated texts are sufficiently complex for the grade level and their intended purpose.

1. It estimates a student’s background knowledge given the selected grade level.
2. It uses the background knowledge estimate as a starting point to evaluate the complexity of a passage’s vocabulary.

### Install & Load necessary packages

In [ ]:
%pip install -qU pydantic textstat langchain langchain_openai langchain-google-genai

In [ ]:
# Load packages
import getpass
import os

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from textstat import textstat as ts
from capture import reset_captures, capture_llm, capture_case, build_contract_toml


### Set up the evaluator's model and prompts

In [ ]:
from prompts import vocab_prompts as prompts

# Set your api keys in your environment, .env file, or enter when prompted.
# os.environ['GOOGLE_API_KEY'] = 'YOUR API KEY'
# os.environ['OPENAI_API_KEY'] = 'YOUR API KEY'
load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

# Grades supported by this evaluator
SUPPORTED_GRADES = range(3, 13)  # 3 through 12 inclusive

VOCAB_TEMPERATURE = 0
# Define the model to be used for vocabulary complexity for grades 3 and 4
VOCAB_MODEL_GRADES_3_4 = "gemini-2.5-pro"
vocab_complexity_model_grades_3_4 = ChatGoogleGenerativeAI(
    model=VOCAB_MODEL_GRADES_3_4, temperature=VOCAB_TEMPERATURE
)

# Define the model to be used for vocabulary complexity for all other grades
VOCAB_MODEL_OTHER_GRADES = "gpt-4.1"
vocab_complexity_model_other_grades = ChatOpenAI(
    model=VOCAB_MODEL_OTHER_GRADES, temperature=VOCAB_TEMPERATURE
)

# Define the model to be used for student background knowledge generation
BK_MODEL = "gpt-4o-2024-11-20"
BK_TEMPERATURE = 0
student_bk_model = ChatOpenAI(model=BK_MODEL, temperature=BK_TEMPERATURE)


### Set up student background knowledge generator

In [ ]:
def get_background_knowledge_assumption(text, grade):
    """Use the background knowledge prompt from the prompts file."""
    prompt = prompts.bk_prompt.format(text=text, grade=grade)

    return capture_llm("background_knowledge", student_bk_model).invoke(prompt).content


### Set up the input variables and output format

In [ ]:
class Output(BaseModel):
    tier_2_words: str = Field(description="List of Tier 2 words")
    tier_3_words: str = Field(description="List of Tier 3 words")
    archaic_words: str = Field(description="List of Archaic words")
    other_complex_words: str = Field(description="List of Other Complex words")
    complexity_score: str = Field(
        description="the complexity of the text, one of: slightly complex, moderately complex, very complex, or exceedingly complex"
    )
    reasoning: str = Field(description="your reasoning for your answer")


prompt_vars = {
    "inputVars": [
        "text",
        "student_grade_level",
        "student_background_knowledge",
        "fk_level",
    ],
    "outputParser": JsonOutputParser(pydantic_object=Output),
}


### Helper functions

In [ ]:
import textwrap


def calculate_fk_score(text) -> float:
    """
    Calculate the Flesch-Kincaid Grade Level
    """
    fk_score = round(ts.flesch_kincaid_grade(text), 2)

    return fk_score


def prepare_text_for_complexity_prediction(text, grade):
    """
    Enrich the text and grade given by user with additional features for complexity prediction.
    """
    dataset = {
        "text": text,
        "student_grade_level": grade,
        "fk_level": calculate_fk_score(text),
        "student_background_knowledge": get_background_knowledge_assumption(
            text, grade
        ),
    }

    return dataset


def prettify_vocab_complexity_output(vocab_complexity_output):
    output = f"""
        ========================= Complexity Score ========================
        {vocab_complexity_output.get('complexity_score') or vocab_complexity_output.get('answer') or 'N/A'}

        ========================= Complexity Score Reasoning ==============
        {textwrap.fill(vocab_complexity_output.get('reasoning', 'N/A'), width=80)}

        ========================  Complex words  ==========================
        * Tier 2 words: {textwrap.fill(vocab_complexity_output.get('tier_2_words', 'N/A'), width=65)}
        * Tier 3 words: {textwrap.fill(vocab_complexity_output.get('tier_3_words', 'N/A'), width=65)}
        * Archaic words: {textwrap.fill(vocab_complexity_output.get('archaic_words', 'N/A'), width=65)}
        * Other complex words: {textwrap.fill(vocab_complexity_output.get('other_complex_words', 'N/A'), width=60)}"""

    print(textwrap.dedent(output).strip())

In [ ]:
def get_prompts_for_grade(grade: int) -> dict:
    """
    Returns the appropriate SYSTEM_PROMPT and USER_PROMPT for the given grade.
    
    Args:
        grade: Grade level (3-12)
    
    Returns:
        dict with keys 'SYSTEM_PROMPT' and 'USER_PROMPT'
    """
    if grade == 3 or grade == 4:
        return prompts.GRADE_SPECIFIC_PROMPTS["GRADES_3_4"]
    else:  # 5-12
        return prompts.GRADE_SPECIFIC_PROMPTS["OTHER_GRADES"]


def get_vocab_model_for_grade(grade: int):
    """
    Returns the appropriate vocabulary complexity model for the given grade.

    Grades 3 & 4 use Gemini (gemini-2.5-pro), which was validated against
    the GRADES_3_4 prompt. All other grades use GPT-4.1, which was validated
    against the OTHER_GRADES prompt.

    Args:
        grade: Grade level (3-12)

    Returns:
        A LangChain chat model instance
    """
    if grade == 3 or grade == 4:
        return vocab_complexity_model_grades_3_4
    else:  # 5-12
        return vocab_complexity_model_other_grades


def normalize_complexity_output(output: dict) -> dict:
    """
    Normalize complexity output to use consistent string labels.
    Converts integer 'answer' (from OTHER_GRADES) to string 'complexity_score'.
    
    Args:
        output: Raw output from the model
    
    Returns:
        Normalized output with 'complexity_score' field
    """
    mapping = {
        1: "Slightly Complex",
        2: "Moderately Complex",
        3: "Very Complex",
        4: "Exceedingly Complex"
    }

    # Handle 'answer' field from OTHER_GRADES (will be int or string int)
    if 'answer' in output:
        value = output['answer']
        # Convert int or string int to proper complexity label
        if isinstance(value, str) and value.isdigit():
            value = int(value)
        output['complexity_score'] = mapping.get(value, str(value))
    
    # For GRADES_3_4, complexity_score already exists as a string - no changes needed

    return output


In [ ]:
def predict_text_complexity_level(text, grade):
    """
    Predict the text complexity level as well as the complex words and reasoning.

    Args:
        text: The text to evaluate.
        grade: Grade level. Must be between 3 and 12 inclusive.

    Raises:
        ValueError: If grade is not in SUPPORTED_GRADES.
    """
    if grade not in SUPPORTED_GRADES:
        raise ValueError(
            f"Grade {grade} is not supported. This evaluator supports grades "
            f"{min(SUPPORTED_GRADES)}-{max(SUPPORTED_GRADES)}."
        )

    dataset = prepare_text_for_complexity_prediction(text, grade)

    # Get grade-specific prompts and model
    grade_prompts = get_prompts_for_grade(grade)

    # Use grade-specific prompts
    messages = [
        SystemMessage(content=grade_prompts['SYSTEM_PROMPT']),
        HumanMessagePromptTemplate.from_template(grade_prompts['USER_PROMPT']),
    ]

    # Prepare chat prompt
    prompt = ChatPromptTemplate(
        messages,
        input_variables=prompt_vars["inputVars"],
        partial_variables={
            "format_instructions": prompt_vars["outputParser"].get_format_instructions()
        },
    )

    # Invoke the chain
    chain = prompt | capture_llm("vocab_complexity", get_vocab_model_for_grade(grade)) | JsonOutputParser()

    # Get output and normalize it
    output = chain.invoke(dataset)
    output = normalize_complexity_output(output)

    return output


# Test out examples

In [ ]:
# Add your text & the grade level you want to evaluate for vocabulary complexity

# Clear ID = 2204
text = """
Polo went on a 24-year trip to China with his father and uncle during the Mongol Dynasty. He left Venice at the age of 17 on a boat that went through the Mediterranean Sea, Ayas, Tabriz and Kerman. Then he travelled across Asia getting as far as Beijing. On the way there he had to go over mountains and through terrible deserts, across hot burning lands and places where the cold was horrible. He served in Kublai Khan's court for 17 years. He left the Far East and returned to Venice by sea. There was sickness on board and 600 passengers and crew died and some say pirates attacked. Nevertheless, Marco Polo survived it all.
Some scholars believe that while Marco Polo did go to China, he did not go to all of the other places described in his book. He brought noodles back from China and the Italians came up with different sizes and shapes and called it pasta. Polo returned to Venice with treasures like ivory, jade, jewels, porcelain and silk.
His father had borrowed money and bought a ship. He became wealthy because of his trading in the near East.
"""

grade_level = 3

vocabulary_complexity_output = predict_text_complexity_level(text, grade_level)

# Pretty Print the output
prettify_vocab_complexity_output(vocabulary_complexity_output)

In [ ]:
reset_captures()
text = """
Polo went on a 24-year trip to China with his father and uncle during the Mongol Dynasty. He left Venice at the age of 17 on a boat that went through the Mediterranean Sea, Ayas, Tabriz and Kerman. Then he travelled across Asia getting as far as Beijing. On the way there he had to go over mountains and through terrible deserts, across hot burning lands and places where the cold was horrible. He served in Kublai Khan's court for 17 years. He left the Far East and returned to Venice by sea. There was sickness on board and 600 passengers and crew died and some say pirates attacked. Nevertheless, Marco Polo survived it all.
Some scholars believe that while Marco Polo did go to China, he did not go to all of the other places described in his book. He brought noodles back from China and the Italians came up with different sizes and shapes and called it pasta. Polo returned to Venice with treasures like ivory, jade, jewels, porcelain and silk.
His father had borrowed money and bought a ship. He became wealthy because of his trading in the near East.
"""
input = {"text": text, "grade": 3}
result = predict_text_complexity_level(**input)

capture = capture_case(
    name="marco_polo_grade3",
    description="Marco Polo passage, grade 3 (grades 3-4 Gemini path)",
    input=input,
    llm_call_captures=["background_knowledge", "vocab_complexity"],
    expected_result=result,
)

print(build_contract_toml(capture))

In [ ]:
reset_captures()
text = """
Great whirling storms roar out of the oceans in many parts of the world. They are called by several names — hurricane, typhoon, and cyclone are the three most familiar ones. But no matter what they are called, they are all the same sort of storm. They are born in the same way, in tropical waters. They develop the same way, feeding on warm, moist air. And they do the same kind of damage, both ashore and at sea. Other storms may cover a bigger area or have higher winds, but none can match both the size and the fury of hurricanes. They are earth's mightiest storms.

Like all storms, they take place in the atmosphere, the envelope of air that surrounds the earth and presses on its surface. The pressure at any one place is always changing. There are days when air is sinking and the atmosphere presses harder on the surface. These are the times of high pressure. There are days when a lot of air is rising and the atmosphere does not press down as hard. These are times of low pressure. Low-pressure areas over warm oceans give birth to hurricanes.
"""
input = {"text": text, "grade": 7}
result = predict_text_complexity_level(**input)

capture = capture_case(
    name="hurricanes_grade7",
    description="Hurricane formation passage, grade 7 (grades 5-12 GPT path)",
    input=input,
    llm_call_captures=["background_knowledge", "vocab_complexity"],
    expected_result=result,
)

print(build_contract_toml(capture))

You can copy or edit the above cell to test out different texts and grade levels.